# StreetForward 演示 Notebook

本 notebook 展示 StreetForward 训练器的基本功能和训练过程。

## 功能概述

### StreetForward
1. Feed-forward 3DGS 训练器，基于代理参数的多视角梯度累积
2. 使用 node_state 作为 detached buffer 存储 Gaussian 参数
3. 通过 MLP 预测偏移量，而不是直接预测参数
4. 使用 Proxy 参数进行渲染，避免二次反传共享图问题
5. 支持多视角监督和梯度回灌机制

### 本 Notebook 包含
1. MultiSceneDataset 数据加载
2. RGB 点云生成
3. Batch 格式转换（MultiSceneDataset → StreetForward）
4. StreetForwardTrainer 初始化和训练
5. 训练循环演示
6. 结果可视化

## 使用说明

1. 按顺序执行所有单元格
2. 在"配置准备"部分修改配置文件路径（如果需要）
3. 每个部分可以独立运行和调试
4. 注意内存使用，特别是点云生成和训练部分

## 第一部分：环境配置和导入

安装和导入所有必要的依赖包。

In [2]:
# 安装依赖（如果需要）
# !pip install numpy matplotlib open3d omegaconf torch

import os
import sys
import numpy as np
import torch
import types
from omegaconf import OmegaConf
from typing import List, Dict, Optional
import matplotlib.pyplot as plt
import open3d as o3d

# 添加项目路径以导入模块
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# 导入项目模块
from datasets.multi_scene_dataset import MultiSceneDataset
from datasets.pointcloud_generators import MonocularRGBPointCloudGenerator
from models.trainers.streetforward import StreetForwardTrainer

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 设置随机种子（可选，用于可重复性）
torch.manual_seed(42)
np.random.seed(42)

print("Environment setup completed!")

/home/a/anaconda3/envs/drivestudio-new/lib/python3.9/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


ImportError: /home/a/anaconda3/envs/drivestudio-new/lib/python3.9/site-packages/pytorch3d/_C.cpython-39-x86_64-linux-gnu.so: undefined symbol: _ZN2at4_ops10zeros_like4callERKNS_6TensorEN3c108optionalINS5_10ScalarTypeEEENS6_INS5_6LayoutEEENS6_INS5_6DeviceEEENS6_IbEENS6_INS5_12MemoryFormatEEE

## 第二部分：配置准备

读取配置文件，准备数据配置和 StreetForward 训练器配置。

In [ ]:
# 读取配置文件（MultiSceneDataset 配置）
config_path = os.path.join(project_root, "configs/evolsplat/multi_scene.yaml")
cfg = OmegaConf.load(config_path)

# 提取数据配置
data_cfg = cfg.data

# 提取 MultiSceneDataset 配置
multi_scene_cfg = cfg.multi_scene

# 显示配置信息
print("Data configuration:")
print(f"  Data root: {data_cfg.data_root}")
print(f"  Dataset type: {data_cfg.dataset}")
print(f"  Train scene IDs: {data_cfg.train_scene_ids}")
print(f"  Eval scene IDs: {data_cfg.eval_scene_ids}")

print("\nMultiSceneDataset configuration:")
print(f"  Num source keyframes: {multi_scene_cfg.num_source_keyframes}")
print(f"  Num target keyframes: {multi_scene_cfg.num_target_keyframes}")
print(f"  Segment overlap ratio: {multi_scene_cfg.segment_overlap_ratio}")
print(f"  Min keyframes per scene: {multi_scene_cfg.min_keyframes_per_scene}")
print(f"  Min keyframes per segment: {multi_scene_cfg.min_keyframes_per_segment}")

# 显示 pointcloud 配置（如果存在）
if hasattr(data_cfg, 'pointcloud'):
    print("\nPoint cloud configuration:")
    print(f"  Sparsity: {data_cfg.pointcloud.get('sparsity', 'N/A')}")
    print(f"  Filter sky: {data_cfg.pointcloud.get('filter_sky', 'N/A')}")
    print(f"  Depth consistency: {data_cfg.pointcloud.get('depth_consistency', 'N/A')}")
    print(f"  Use bounding box: {data_cfg.pointcloud.get('use_bbx', 'N/A')}")
    print(f"  Downscale: {data_cfg.pointcloud.get('downscale', 'N/A')}")

# 准备 fixed_segment_aabb（如果配置了）
fixed_segment_aabb = None
if multi_scene_cfg.fixed_segment_aabb is not None:
    fixed_segment_aabb = torch.tensor(multi_scene_cfg.fixed_segment_aabb, dtype=torch.float32)
    print(f"\nUsing fixed segment AABB: {fixed_segment_aabb}")

In [ ]:
# 创建 StreetForward 训练器配置
# 参考 tests/test_streetforward.py 和模型设计文档

# 从 segment AABB 获取 bbx_min 和 bbx_max（如果有固定AABB）
if fixed_segment_aabb is not None:
    bbx_min = fixed_segment_aabb[0].tolist()
    bbx_max = fixed_segment_aabb[1].tolist()
else:
    # 使用默认值（可以从配置文件中读取）
    bbx_min = [-20.0, -20.0, -20.0]
    bbx_max = [20.0, 4.8, 70.0]

streetforward_config = OmegaConf.create({
    "model": {
        "sparseConv_outdim": 32,  # 3D特征维度
        "offset_max": 0.1,  # 位置偏移最大值
        "sh_degree": 1,  # SH 阶数
        "voxel_size": 0.1,  # 体素大小
        "max_iterations": 1,  # 内部迭代次数（每次train_iter内部的迭代）
        "bbx_min": bbx_min,
        "bbx_max": bbx_max,
    },
    "optimizer": {
        "lr": 1e-3,
        "eps": 1e-15,
        "weight_decay": 0.0,
    },
    "log_images": False,  # 是否保存渲染图像（会占用更多GPU内存）
})

print("StreetForward configuration:")
print(f"  SparseConv output dim: {streetforward_config.model.sparseConv_outdim}")
print(f"  Offset max: {streetforward_config.model.offset_max}")
print(f"  SH degree: {streetforward_config.model.sh_degree}")
print(f"  Voxel size: {streetforward_config.model.voxel_size}")
print(f"  Max iterations: {streetforward_config.model.max_iterations}")
print(f"  Bounding box min: {streetforward_config.model.bbx_min}")
print(f"  Bounding box max: {streetforward_config.model.bbx_max}")
print(f"  Learning rate: {streetforward_config.optimizer.lr}")

## 第三部分：MultiSceneDataset 初始化

创建 MultiSceneDataset 实例并初始化数据集。

In [ ]:
# 创建 MultiSceneDataset 实例
dataset = MultiSceneDataset(
    data_cfg=data_cfg,
    train_scene_ids=data_cfg.train_scene_ids,
    eval_scene_ids=data_cfg.eval_scene_ids,
    num_source_keyframes=multi_scene_cfg.num_source_keyframes,
    num_target_keyframes=multi_scene_cfg.num_target_keyframes,
    segment_overlap_ratio=multi_scene_cfg.segment_overlap_ratio,
    keyframe_split_config=dict(multi_scene_cfg.keyframe_split_config),
    min_keyframes_per_scene=multi_scene_cfg.min_keyframes_per_scene,
    min_keyframes_per_segment=multi_scene_cfg.min_keyframes_per_segment,
    device=device,
    preload_scene_count=2,  # 预加载2个场景（减少内存占用）
    fixed_segment_aabb=fixed_segment_aabb,
)

print("MultiSceneDataset created successfully!")

In [ ]:
# 初始化数据集（可选，会在第一次使用时自动初始化）
dataset.initialize()

# 获取当前场景ID
current_scene_id = dataset.get_current_scene_id()
print(f"Current training scene ID: {current_scene_id}")

# 获取场景信息（如果场景已加载）
if current_scene_id is not None:
    scene_info = dataset.get_scene(current_scene_id)
    if scene_info:
        print(f"\nScene {current_scene_id} information:")
        print(f"  Number of segments: {len(scene_info['segments'])}")
        print(f"  Number of frames: {scene_info['num_frames']}")
        print(f"  Number of cameras: {scene_info['num_cams']}")
        print(f"  Number of keyframe segments: {len(scene_info['keyframe_segments'])}")
        
        # 显示每个段的信息
        for i, segment in enumerate(scene_info['segments']):
            print(f"\n  Segment {i}:")
            print(f"    Keyframe indices: {segment['keyframe_indices']}")
            print(f"    Number of frames: {len(segment['frame_indices'])}")
            print(f"    AABB shape: {segment['aabb'].shape}")

## 第四部分：点云生成

创建 RGBPointCloudGenerator 并为指定场景和段生成点云。

In [ ]:
# 创建 RGB 点云生成器
# 从配置文件中获取参数
crop_aabb = np.array(data_cfg.pointcloud.crop_aabb) if hasattr(data_cfg, 'pointcloud') else None
input_aabb = np.array(data_cfg.pointcloud.input_aabb) if hasattr(data_cfg, 'pointcloud') else None

pointcloud_generator = MonocularRGBPointCloudGenerator(
    chosen_cam_ids=data_cfg.pixel_source.cameras,
    sparsity=data_cfg.pointcloud.get("sparsity", "full") if hasattr(data_cfg, 'pointcloud') else "full",
    filter_sky=data_cfg.pointcloud.get("filter_sky", True) if hasattr(data_cfg, 'pointcloud') else True,
    depth_consistency=data_cfg.pointcloud.get("depth_consistency", True) if hasattr(data_cfg, 'pointcloud') else True,
    use_bbx=data_cfg.pointcloud.get("use_bbx", True) if hasattr(data_cfg, 'pointcloud') else True,
    downscale=data_cfg.pointcloud.get("downscale", 2) if hasattr(data_cfg, 'pointcloud') else 2,
    crop_aabb=crop_aabb,
    input_aabb=input_aabb,
    device=device,
)

print("RGBPointCloudGenerator created successfully!")
print(f"  Chosen camera IDs: {pointcloud_generator.chosen_cam_ids}")
print(f"  Sparsity: {pointcloud_generator.sparsity}")
print(f"  Filter sky: {pointcloud_generator.filter_sky}")
print(f"  Depth consistency: {pointcloud_generator.depth_consistency}")
print(f"  Use bounding box: {pointcloud_generator.use_bbx}")
print(f"  Downscale: {pointcloud_generator.downscale}")

In [ ]:
# 为指定场景和段生成点云
# 使用第一个场景的第一个段作为示例

scene_id = dataset.get_current_scene_id()
if scene_id is None:
    print("No scene available. Please check dataset initialization.")
else:
    scene_info = dataset.get_scene(scene_id)
    if scene_info and len(scene_info['segments']) > 0:
        segment_id = 0
        print(f"Generating point cloud for scene {scene_id}, segment {segment_id}...")
        
        try:
            pointcloud_dict = pointcloud_generator.generate_pointcloud(
                dataset=dataset,
                scene_id=scene_id,
                segment_id=segment_id,
            )
            
            # 点云生成器返回的是一个字典，包含 "background" 键
            # background: np.ndarray [N, 6] (xyz + rgb)
            background_points = pointcloud_dict.get("background", np.zeros((0, 6), dtype=np.float32))
            
            print(f"\nPoint cloud generated successfully!")
            print(f"  Background points shape: {background_points.shape}")
            print(f"  Number of background points: {len(background_points)}")
            
            if len(background_points) > 0:
                print(f"  Points range (xyz):")
                print(f"    X: [{background_points[:, 0].min():.2f}, {background_points[:, 0].max():.2f}]")
                print(f"    Y: [{background_points[:, 1].min():.2f}, {background_points[:, 1].max():.2f}]")
                print(f"    Z: [{background_points[:, 2].min():.2f}, {background_points[:, 2].max():.2f}]")
                print(f"  Colors range (rgb):")
                print(f"    R: [{background_points[:, 3].min():.2f}, {background_points[:, 3].max():.2f}]")
                print(f"    G: [{background_points[:, 4].min():.2f}, {background_points[:, 4].max():.2f}]")
                print(f"    B: [{background_points[:, 5].min():.2f}, {background_points[:, 5].max():.2f}]")
            
            # 检查是否有动态物体
            dynamic_objects = pointcloud_dict.get("dynamic_objects", {})
            if dynamic_objects:
                print(f"  Dynamic objects: {len(dynamic_objects)} instances")
                for intid, pts in dynamic_objects.items():
                    print(f"    Instance {intid}: {len(pts)} points")
            
        except Exception as e:
            print(f"Error generating point cloud: {e}")
            import traceback
            traceback.print_exc()
            pointcloud_dict = None
    else:
        print("No segments available in current scene.")
        pointcloud_dict = None

## 第五部分：Batch 格式转换

创建辅助函数将 MultiSceneDataset 的 batch 转换为 StreetForward 需要的格式。

In [ ]:
# 定义 DummyView 类（参考 test_streetforward.py）
class DummyView(types.SimpleNamespace):
    """Simple container to mimic camera objects used by the trainer."""
    pass


def convert_to_streetforward_batch(multi_scene_batch, pointcloud_dict, device):
    """
    将 MultiSceneDataset 的 batch 转换为 StreetForward 格式
    
    Args:
        multi_scene_batch: MultiSceneDataset 返回的 batch
        pointcloud_dict: 点云字典，包含 "background" 键（np.ndarray [N, 6]）
        device: torch.device
    
    Returns:
        StreetForward batch format:
        {
            'scene_id': int,
            'segment_id': int,
            'pointcloud': dict with 'background' key,
            'target_views': list of view objects,
            'gt_images': list of images
        }
    """
    # 1. 提取 scene_id 和 segment_id
    scene_id = multi_scene_batch['scene_id'].item() if isinstance(multi_scene_batch['scene_id'], torch.Tensor) else multi_scene_batch['scene_id']
    segment_id = multi_scene_batch['segment_id']
    
    # 2. 转换点云格式（pointcloud_dict 已经是正确的格式）
    # StreetForward 期望: {"background": np.ndarray [N, 6]}
    pointcloud_formatted = {"background": pointcloud_dict["background"]}
    
    # 3. 从 target 数据创建 view 对象和 gt_images
    target_extrinsics = multi_scene_batch['target']['extrinsics']  # [N, 4, 4]
    target_intrinsics = multi_scene_batch['target']['intrinsics']  # [N, 4, 4]
    target_images = multi_scene_batch['target']['image']  # [N, H, W, 3]
    
    target_views = []
    gt_images = []
    
    for i in range(target_extrinsics.shape[0]):
        c2w = target_extrinsics[i].unsqueeze(0).to(device)  # [1, 4, 4]
        k_4x4 = target_intrinsics[i].to(device)  # [4, 4]
        k_3x3 = k_4x4[:3, :3].unsqueeze(0)  # [1, 3, 3]
        
        view = DummyView(camtoworlds=c2w, Ks=k_3x3)
        target_views.append(view)
        gt_images.append(target_images[i].to(device))  # [H, W, 3]
    
    return {
        "scene_id": scene_id,
        "segment_id": segment_id,
        "pointcloud": pointcloud_formatted,
        "target_views": target_views,
        "gt_images": gt_images,
    }

print("Batch conversion function defined successfully!")

## 第六部分：StreetForwardTrainer 初始化

创建 StreetForwardTrainer 实例并展示模型结构。

In [ ]:
# 创建 StreetForwardTrainer 实例
trainer = StreetForwardTrainer(
    config=streetforward_config,
    device=device,
)

print("StreetForwardTrainer created successfully!")
print(f"\nModel structure:")
print(f"  SparseConv: {type(trainer.sparse_conv).__name__}")
print(f"  MLP Offset Position: {trainer.mlp_offset_pos}")
print(f"  MLP Conv (scales + quats): {trainer.mlp_conv}")
print(f"  MLP Opacity: {trainer.mlp_opacity}")
print(f"  Gaussian Decoder (SH): {trainer.gaussion_decoder}")

# 计算参数量
total_params = sum(p.numel() for p in trainer.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

# 显示优化器配置
print(f"\nOptimizer:")
print(f"  Type: {type(trainer.optimizer).__name__}")
print(f"  Learning rate: {trainer.optimizer.param_groups[0]['lr']}")
print(f"  Eps: {trainer.optimizer.param_groups[0]['eps']}")
print(f"  Weight decay: {trainer.optimizer.param_groups[0]['weight_decay']}")

# 检查节点状态字典（初始为空）
print(f"\nNode states (initial): {len(trainer.node_states)} nodes")

## 第七部分：单个 Batch 训练演示

获取一个 batch，生成点云，转换格式，运行 train_iter()。

In [ ]:
# 获取一个 batch（从 MultiSceneDataset）
scene_id = dataset.get_current_scene_id()
if scene_id is None:
    print("No scene available. Please check dataset initialization.")
else:
    scene_info = dataset.get_scene(scene_id)
    if scene_info and len(scene_info['segments']) > 0:
        segment_id = 0
        
        print(f"Getting batch for scene {scene_id}, segment {segment_id}...")
        multi_scene_batch = dataset.get_segment_batch(scene_id=scene_id, segment_id=segment_id)
        
        print(f"\nMultiSceneDataset batch structure:")
        print(f"  Scene ID: {multi_scene_batch['scene_id']}")
        print(f"  Segment ID: {multi_scene_batch['segment_id']}")
        print(f"  Source images shape: {multi_scene_batch['source']['image'].shape}")
        print(f"  Target images shape: {multi_scene_batch['target']['image'].shape}")
        print(f"  Target extrinsics shape: {multi_scene_batch['target']['extrinsics'].shape}")
        print(f"  Target intrinsics shape: {multi_scene_batch['target']['intrinsics'].shape}")
        
        # 生成点云
        print(f"\nGenerating point cloud...")
        pointcloud_dict = pointcloud_generator.generate_pointcloud(
            dataset=dataset,
            scene_id=scene_id,
            segment_id=segment_id,
        )
        print(f"  Background points: {pointcloud_dict['background'].shape}")
        
        # 转换 batch 格式
        print(f"\nConverting batch format...")
        streetforward_batch = convert_to_streetforward_batch(
            multi_scene_batch=multi_scene_batch,
            pointcloud_dict=pointcloud_dict,
            device=device,
        )
        
        print(f"\nStreetForward batch structure:")
        print(f"  Scene ID: {streetforward_batch['scene_id']}")
        print(f"  Segment ID: {streetforward_batch['segment_id']}")
        print(f"  Pointcloud background shape: {streetforward_batch['pointcloud']['background'].shape}")
        print(f"  Number of target views: {len(streetforward_batch['target_views'])}")
        print(f"  Number of GT images: {len(streetforward_batch['gt_images'])}")
        if len(streetforward_batch['gt_images']) > 0:
            print(f"  GT image shape: {streetforward_batch['gt_images'][0].shape}")
    else:
        print("No segments available in current scene.")
        streetforward_batch = None

In [ ]:
# 运行 train_iter()（不更新状态，仅演示）
if 'streetforward_batch' in locals() and streetforward_batch is not None:
    print("Running train_iter() (without update)...")
    
    # 运行一次迭代（不更新状态，仅演示前向传播和损失计算）
    # 注意：虽然 apply_update=False 和 update_state=False，但仍需要梯度来计算损失
    try:
        outputs = trainer.train_iter(
            batch=streetforward_batch,
            apply_update=False,  # 不更新优化器
            update_state=False,  # 不更新 node_state
        )
        
        print(f"\nTrain iteration completed!")
        print(f"  Total loss: {outputs['total_loss'].item():.6f}")
        print(f"  Number of outputs: {len(outputs['outputs'])}")
        
        if len(outputs['outputs']) > 0:
            print(f"\n  First output:")
            first_output = outputs['outputs'][0]
            if 'loss' in first_output:
                print(f"    Loss: {first_output['loss']:.6f}")
            if 'rgb' in first_output:
                print(f"    RGB shape: {first_output['rgb'].shape}")
            if 'acc' in first_output:
                print(f"    Accumulation shape: {first_output['acc'].shape}")
        
        # 检查节点状态（应该已创建）
        key = (streetforward_batch['scene_id'], streetforward_batch['segment_id'])
        if key in trainer.node_states:
            node_state = trainer.node_states[key]
            print(f"\n  Node state (scene {streetforward_batch['scene_id']}, segment {streetforward_batch['segment_id']}):")
            print(f"    Means shape: {node_state.means.shape}")
            print(f"    Scales log shape: {node_state.scales_log.shape}")
            print(f"    Quats shape: {node_state.quats.shape}")
            print(f"    Opacity logit shape: {node_state.opacity_logit.shape}")
            print(f"    SH DC shape: {node_state.sh_dc.shape}")
            print(f"    SH rest shape: {node_state.sh_rest.shape}")
        
    except Exception as e:
        print(f"Error during train_iter: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No batch available. Please run the previous cell first.")

## 第八部分：训练循环演示

使用调度器进行简化的训练循环，展示损失变化。

In [ ]:
# 创建调度器
scheduler = dataset.create_scheduler(
    batches_per_segment=5,  # 每个段遍历5次（演示用，实际训练可以更多）
    segment_order="random",
    scene_order="random",
    shuffle_segments=True,
    preload_next_scene=True,
)

print("Scheduler created successfully!")
print(f"  Batches per segment: 5")
print(f"  Segment order: random")
print(f"  Scene order: random")

In [ ]:
# 简化的训练循环（少量迭代，仅演示）
num_iterations = 3  # 演示用，只运行3次迭代

losses = []
scene_ids_list = []
segment_ids_list = []

print(f"Starting training loop ({num_iterations} iterations)...")
print("-" * 80)

try:
    for iteration in range(num_iterations):
        # 获取下一个 batch
        try:
            multi_scene_batch = scheduler.next_batch()
        except StopIteration:
            print("All scenes processed. Resetting scheduler...")
            scheduler.reset()
            multi_scene_batch = scheduler.next_batch()
        
        scene_id = multi_scene_batch['scene_id'].item() if isinstance(multi_scene_batch['scene_id'], torch.Tensor) else multi_scene_batch['scene_id']
        segment_id = multi_scene_batch['segment_id']
        
        # 获取当前状态信息
        info = scheduler.get_current_info()
        
        print(f"\nIteration {iteration + 1}/{num_iterations}:")
        print(f"  Scene ID: {scene_id}, Segment ID: {segment_id}")
        print(f"  Batch count: {info['batch_count']}/{info['batches_per_segment']}")
        
        # 生成点云
        pointcloud_dict = pointcloud_generator.generate_pointcloud(
            dataset=dataset,
            scene_id=scene_id,
            segment_id=segment_id,
        )
        
        # 转换 batch 格式
        streetforward_batch = convert_to_streetforward_batch(
            multi_scene_batch=multi_scene_batch,
            pointcloud_dict=pointcloud_dict,
            device=device,
        )
        
        # 运行训练迭代（这次会更新状态）
        outputs = trainer.train_iter(
            batch=streetforward_batch,
            apply_update=True,  # 更新优化器
            update_state=True,  # 更新 node_state
        )
        
        loss = outputs['total_loss'].item()
        losses.append(loss)
        scene_ids_list.append(scene_id)
        segment_ids_list.append(segment_id)
        
        print(f"  Loss: {loss:.6f}")
        print(f"  Number of target views: {len(streetforward_batch['target_views'])}")
        
        # 释放内存（可选）
        del multi_scene_batch, pointcloud_dict, streetforward_batch, outputs
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

except KeyboardInterrupt:
    print("\nTraining interrupted by user.")
except Exception as e:
    print(f"\nError during training: {e}")
    import traceback
    traceback.print_exc()
finally:
    # 清理调度器
    scheduler.shutdown()

print("\n" + "-" * 80)
print("Training loop completed!")
print(f"  Total iterations: {len(losses)}")
print(f"  Final loss: {losses[-1]:.6f if losses else 'N/A'}")
print(f"  Average loss: {np.mean(losses):.6f if losses else 'N/A'}")
print(f"  Min loss: {np.min(losses):.6f if losses else 'N/A'}")
print(f"  Max loss: {np.max(losses):.6f if losses else 'N/A'}")

## 第九部分：可视化（可选）

展示训练过程中的损失曲线和节点状态信息。

In [ ]:
# 绘制损失曲线
if len(losses) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(losses, marker='o', linestyle='-', linewidth=2, markersize=8)
    plt.xlabel('Iteration', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training Loss Curve', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nLoss statistics:")
    print(f"  Iterations: {len(losses)}")
    print(f"  Final loss: {losses[-1]:.6f}")
    print(f"  Average loss: {np.mean(losses):.6f}")
    print(f"  Std loss: {np.std(losses):.6f}")
else:
    print("No loss data available. Please run the training loop first.")

In [ ]:
# 展示节点状态信息
print("Node states summary:")
print(f"  Total number of nodes: {len(trainer.node_states)}")
print("\nNode details:")

for (scene_id, segment_id), node_state in trainer.node_states.items():
    print(f"\n  Scene {scene_id}, Segment {segment_id}:")
    print(f"    Number of Gaussians: {node_state.means.shape[0]}")
    print(f"    Means range:")
    print(f"      X: [{node_state.means[:, 0].min().item():.2f}, {node_state.means[:, 0].max().item():.2f}]")
    print(f"      Y: [{node_state.means[:, 1].min().item():.2f}, {node_state.means[:, 1].max().item():.2f}]")
    print(f"      Z: [{node_state.means[:, 2].min().item():.2f}, {node_state.means[:, 2].max().item():.2f}]")
    print(f"    Scales log range: [{node_state.scales_log.min().item():.4f}, {node_state.scales_log.max().item():.4f}]")
    print(f"    Opacity logit range: [{node_state.opacity_logit.min().item():.4f}, {node_state.opacity_logit.max().item():.4f}]")

In [ ]:
# 可选：可视化点云（如果点云不太大）
# 注意：如果点云很大，可视化可能会很慢

if 'pointcloud_dict' in locals() and pointcloud_dict is not None:
    background_points = pointcloud_dict.get("background", np.zeros((0, 6), dtype=np.float32))
    
    if len(background_points) > 0 and len(background_points) < 100000:  # 只可视化小于10万个点的点云
        print(f"\nVisualizing point cloud ({len(background_points)} points)...")
        
        # 创建 Open3D 点云对象
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(background_points[:, :3])
        
        # 转换颜色到 [0, 1] 范围（如果原始颜色在 [0, 255] 范围）
        colors = background_points[:, 3:6].copy()
        if colors.max() > 1.0 + 1e-3:
            colors = colors / 255.0
        colors = np.clip(colors, 0.0, 1.0)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        
        # 可视化
        print("  Displaying point cloud...")
        o3d.visualization.draw_geometries([pcd], window_name="Point Cloud Visualization")
    elif len(background_points) >= 100000:
        print(f"\nPoint cloud too large ({len(background_points)} points) for visualization.")
        print("  Consider downsampling or using a smaller segment.")
else:
    print("\nNo point cloud available for visualization.")